In [1]:
import ee
import geemap
import pandas as pd
import numpy as np
from tqdm import tqdm

In [2]:
ee.Authenticate()
ee.Initialize(project='soil-health-project-497508')
print("Earth Engine Connected")

Earth Engine Connected


In [3]:
df = pd.read_csv("cleaned_MH_OC.csv")
print(df.shape)
df.head()

(29942, 161)


,lat_key,fid,latitude,longitude,CLIMATE_VALUE,CLIMATE_SUBCLASS,CLIMATE_CLASS,DOMSOI,SOIL_TYPE,SOIL_SUBCLASS,...,S1_VH_ASM,S1_VH_Energy,S1_VH_MaxProbability,S1_VH_Entropy,S1_VH_GLCM_Mean,S1_VH_GLCM_Variance,S1_VH_GLCM_Correlation,year,month,dayofyear
0,15.709677,48980.0,15.709677,74.047631,2.0,Am,Tropical,Ap,ACRISOLS,Plinthic Acrisols,...,0.699068,0.999988,0.995973,-4.731746,0.022191,0.000029,0.424396,2025,2,48
1,15.709731,48981.0,15.709731,74.048897,2.0,Am,Tropical,Ap,ACRISOLS,Plinthic Acrisols,...,0.636399,0.999990,0.997320,-4.856527,0.016932,0.000098,0.181971,2025,2,48
2,15.711065,48982.0,15.711065,74.047657,2.0,Am,Tropical,Ap,ACRISOLS,Plinthic Acrisols,...,0.606309,0.999960,0.995374,-4.144549,0.013338,0.000111,0.318890,2025,2,48
3,15.804995,48839.5,15.804995,73.733783,2.0,Am,Tropical,Nd,NITOSOLS,Dystric Nitosols,...,0.774612,0.999999,0.998807,-6.255501,0.007464,0.000004,0.314625,2024,3,91
4,15.805012,48835.0,15.805012,73.734112,2.0,Am,Tropical,Nd,NITOSOLS,Dystric Nitosols,...,0.648962,0.999997,0.998350,-5.404119,0.007534,0.000008,1.216725,2024,3,91


In [4]:
embed_input = df[['fid','latitude','longitude','year']].copy()
embed_input.head()

,fid,latitude,longitude,year
0,48980.0,15.709677,74.047631,2025
1,48981.0,15.709731,74.048897,2025
2,48982.0,15.711065,74.047657,2025
3,48839.5,15.804995,73.733783,2024
4,48835.0,15.805012,73.734112,2024


In [5]:
embeddings = ee.ImageCollection("GOOGLE/SATELLITE_EMBEDDING/V1/ANNUAL")
print(embeddings.size().getInfo())

97155


In [6]:
batch_size = 1000

all_results = []

years_to_extract = range(2017, 2026)

for emb_year in years_to_extract:

    print(f"\nProcessing embedding year {emb_year}")

    img = embeddings.filterDate(
        f"{emb_year}-01-01",
        f"{emb_year}-12-31"
    ).mosaic()

    for start in tqdm(
        range(0, len(embed_input), batch_size)
    ):

        end = start + batch_size

        batch = embed_input.iloc[start:end]

        features = []

        for _, row in batch.iterrows():

            features.append(
                ee.Feature(
                    ee.Geometry.Point([
                        row.longitude,
                        row.latitude
                    ]),
                    {
                        "fid": row.fid
                    }
                )
            )

        fc = ee.FeatureCollection(features)

        sampled = img.sampleRegions(
            collection=fc,
            scale=100,
            geometries=False
        )

        features_out = sampled.getInfo()['features']

        rows = [
            f['properties']
            for f in features_out
        ]

        temp = pd.DataFrame(rows)

        temp["embedding_year"] = emb_year

        all_results.append(temp)

embed_df = pd.concat(
    all_results,
    ignore_index=True
)

print(embed_df.shape)

embed_df.to_csv(
    "alphaearth_embeddings_2017_2025.csv",
    index=False
)

print("Saved successfully")


Processing embedding year 2017


100%|██████████| 30/30 [06:24<00:00, 12.82s/it]



Processing embedding year 2018


100%|██████████| 30/30 [07:21<00:00, 14.73s/it]



Processing embedding year 2019


...


Processing embedding year 2024


100%|██████████| 30/30 [03:44<00:00,  7.48s/it]



Processing embedding year 2025


100%|██████████| 30/30 [04:05<00:00,  8.19s/it]


(269478, 66)
Saved successfully
